# YOLO Object Detection — From Scratch

This project is a hands-on implementation of an object detection pipeline using YOLO.

## What I learned

- How YOLO object detection works
- YOLO dataset structure
- YOLO annotation format
- Creating a custom object detection dataset
- Training a YOLO model
- Validating the trained model
- Running predictions on unseen images
- Understanding confidence scores and bounding boxes

## Project

A small synthetic dataset containing two object classes:

- Circle
- Square

The purpose of this project is to understand the complete YOLO workflow before applying it to real-world Side-Scan Sonar marine debris detection.

In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 814.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 1.5 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

print("YOLO is ready!")

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import random
import shutil

# Dataset location
base = Path("/content/mini_dataset")

# Remove old dataset if it exists
if base.exists():
    shutil.rmtree(base)

# Create required YOLO folders
for folder in [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]:
    (base / folder).mkdir(parents=True, exist_ok=True)

print("Dataset folders created.")

In [ ]:
for i in range(20):

    # Create white background
    img = np.ones((640, 640, 3), dtype=np.uint8) * 255

    # Randomly select object class
    cls = random.randint(0, 1)

    # Random object size and position
    size = random.randint(40, 70)

    x = random.randint(size + 20, 640 - size - 20)
    y = random.randint(size + 20, 640 - size - 20)

    # Draw object
    if cls == 0:
        cv2.circle(img, (x, y), size, (0, 0, 0), -1)
    else:
        cv2.rectangle(
            img,
            (x - size, y - size),
            (x + size, y + size),
            (0, 0, 0),
            -1
        )

    # Bounding box in pixel coordinates
    xmin = x - size
    ymin = y - size
    xmax = x + size
    ymax = y + size

    # Convert bounding box to YOLO format
    xc = ((xmin + xmax) / 2) / 640
    yc = ((ymin + ymax) / 2) / 640
    w = (xmax - xmin) / 640
    h = (ymax - ymin) / 640

    # First 16 images → training
    # Last 4 images → validation
    split = "train" if i < 16 else "val"

    # Save image
    image_path = base / f"images/{split}/image_{i}.jpg"
    cv2.imwrite(str(image_path), img)

    # Save YOLO annotation
    label_path = base / f"labels/{split}/image_{i}.txt"

    with open(label_path, "w") as f:
        f.write(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

print("20 images and YOLO annotations created.")

In [ ]:
yaml_content = """
path: /content/mini_dataset

train: images/train
val: images/val

names:
  0: circle
  1: square
"""

with open(base / "data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml created.")

In [ ]:
model = YOLO("yolo11n.pt")

results = model.train(
    data="/content/mini_dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=4,
    patience=100
)

In [ ]:
from pathlib import Path

models = sorted(
    Path("/content/runs/detect").glob("*/weights/best.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

best_model = models[0]

print("Best model:")
print(best_model)

In [ ]:
model = YOLO(str(best_model))

results = model.predict(
    source="/content/mini_dataset/images/val",
    conf=0.25,
    save=True
)

print("Prediction completed.")

In [ ]:
from IPython.display import Image, display

prediction_dir = Path(results[0].save_dir)

for image_path in sorted(prediction_dir.glob("*.jpg")):
    display(Image(filename=str(image_path)))

# Conclusion

This project demonstrated the complete object detection workflow using YOLO:

1. Created a custom dataset
2. Generated YOLO-format annotations
3. Defined dataset classes using `data.yaml`
4. Trained a YOLO object detection model
5. Validated the model on unseen images
6. Generated predictions with confidence scores
7. Visualized predicted bounding boxes

This mini-project establishes the fundamentals required for applying YOLO to real-world object detection problems such as Side-Scan Sonar marine debris detection.